<a href="https://colab.research.google.com/github/junkyuhufs/Class2026spring/blob/main/NLPforET_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gradio Apps for English Teachers 🚀
## This Week: Turning Python Functions into Apps

이번 주에는 **Gradio**를 이용해서  
파이썬으로 만든 기능을 **앱(app)** 형태로 바꾸는 것을 시연합니다.

---

## 이번 주 수업 목표 🎯

이번 수업에서는 아래 3가지를 이해하면 충분합니다.

1. 파이썬 함수가 앱이 될 수 있다는 것  
2. 앱은 기본적으로 **입력 → 함수 → 출력** 구조라는 것  
3. 텍스트, 소리, 이미지, 그림책 퀴즈도 앱으로 만들 수 있다는 것

---

## 오늘 다룰 앱들 📚🎧🖼️

1. **Reading Passage Analyzer**  
2. **Text-to-Speech App**  
3. **Image Processing App**  
4. **Picture Book App**  
5. **Picture Book Order Quiz App**

# App 1. Reading Passage Analyzer 📚

이 앱은 영어 지문을 입력하면 아래를 보여줍니다.

- 문장 수
- 단어 수
- 평균 문장 길이

이 앱을 통해 학생들은  
**텍스트도 분석 가능한 데이터**라는 것을 느낄 수 있습니다.

## 1. Gradio란? 💡

**Gradio**는 파이썬 함수 하나를  
빠르게 웹앱처럼 보여 주게 해 주는 도구입니다.

즉, 아래 구조를 아주 쉽게 앱으로 바꿔 줍니다.

- **입력(input)**  
- **함수(function)**  
- **출력(output)**  

오늘은 이 구조가 실제 앱 화면으로 어떻게 보이는지 체험해 봅니다.

In [ ]:
# =========================================================
# FIXED VERSION: Reading Passage Analyzer
# =========================================================

# 1) 필요한 라이브러리 설치
!pip -q install gradio nltk

# 2) 라이브러리 불러오기
import gradio as gr
import nltk

# ---------------------------------------------------------
# 3) NLTK 자료 다운로드
# ---------------------------------------------------------
# 코랩/최신 nltk에서는 punkt만으로 부족한 경우가 있어서
# punkt_tab도 같이 받아 두는 것이 더 안전합니다.
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.tokenize import sent_tokenize, word_tokenize

# ---------------------------------------------------------
# 4) 분석 함수 만들기
# ---------------------------------------------------------
def analyze_reading(text):
    """
    입력된 영어 지문을 분석하는 함수

    입력(input):
        - text: 영어 지문

    처리(process):
        - 문장 수 세기
        - 단어 수 세기
        - 평균 문장 길이 계산

    출력(output):
        - 문장 수
        - 단어 수
        - 평균 문장 길이
    """

    # 아무 것도 입력하지 않은 경우
    if not text.strip():
        return 0, 0, 0

    try:
        # 문장 단위로 나누기
        sentences = sent_tokenize(text)

        # 단어 단위로 나누고, 알파벳 단어만 남기기
        words = [w for w in word_tokenize(text) if w.isalpha()]

        # 결과 계산
        num_sentences = len(sentences)
        num_words = len(words)
        avg_sentence_length = round(num_words / num_sentences, 2) if num_sentences > 0 else 0

        return num_sentences, num_words, avg_sentence_length

    except Exception as e:
        # 디버깅용: 에러가 나면 콘솔에 출력
        print("Error inside analyze_reading:", e)

        # 앱이 완전히 죽지 않도록 기본값 반환
        return 0, 0, 0

# ---------------------------------------------------------
# 5) Gradio 앱 만들기
# ---------------------------------------------------------
demo = gr.Interface(
    fn=analyze_reading,
    inputs=gr.Textbox(
        lines=10,
        label="Paste a reading passage here"
    ),
    outputs=[
        gr.Number(label="Number of Sentences"),
        gr.Number(label="Number of Words"),
        gr.Number(label="Average Sentence Length")
    ],
    title="Reading Passage Analyzer 📚",
    description="Paste a reading passage and get simple reading statistics."
)

# ---------------------------------------------------------
# 6) 앱 실행
# ---------------------------------------------------------
demo.launch()

### 결과가 의미하는 것 💡

이 앱은 영어 지문을 숫자로 바꾸어 보여줍니다.

즉,
- 텍스트를 그냥 읽는 것에서 끝나는 것이 아니라
- **분석 가능한 데이터**로 볼 수 있다는 뜻입니다.

# App 2. Text-to-Speech App 🎧

이 앱은 입력한 영어 문장을  
영어 음성(mp3)으로 바꿔 줍니다.

즉,  
**텍스트 → 소리**  
로 바꾸는 예시입니다.

In [ ]:
# =========================================================
# APP 2. Text-to-Speech App
# =========================================================

# 1) 필요한 라이브러리 설치
!pip -q install gradio gTTS

# 2) 라이브러리 불러오기
import gradio as gr
from gtts import gTTS
import tempfile

# 3) TTS 함수 만들기
def make_tts(text):
    """
    입력된 영어 문장을 음성 파일로 바꾸는 함수

    입력(input):
        - 영어 문장(text)

    처리(process):
        - gTTS로 텍스트를 음성(mp3)으로 바꾸기

    출력(output):
        - mp3 파일
        - 설명 문장
    """

    # 빈 입력 처리
    if not text.strip():
        return None, "Please enter some text."

    # 임시 mp3 파일 만들기
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    temp_file.close()

    # gTTS로 음성 생성
    tts = gTTS(text=text, lang="en")
    tts.save(temp_file.name)

    message = (
        "Done! ✅\n\n"
        "Input → English text\n"
        "Function → gTTS\n"
        "Output → audio file"
    )

    return temp_file.name, message

# 4) Gradio 앱 만들기
demo = gr.Interface(
    fn=make_tts,
    inputs=gr.Textbox(lines=5, label="Enter English text"),
    outputs=[
        gr.Audio(label="TTS Output", type="filepath"),
        gr.Textbox(label="Explanation")
    ],
    title="Text-to-Speech App 🎧",
    description="Enter English text and convert it into speech."
)

# 5) 앱 실행
demo.launch()

# App 3. Image Processing App 🖼️

이 앱은 이미지를 업로드하면 아래 결과를 보여줍니다.

- 원본 이미지
- 흑백 이미지
- 텍스트가 들어간 이미지

즉,  
**이미지도 입력 데이터가 될 수 있다**는 것을 보여 줍니다.

In [ ]:
# =========================================================
# APP 3. Image Processing App
# Robust caption font version
# =========================================================

# 1) 필요한 라이브러리 설치
!pip -q install gradio pillow

# 2) 라이브러리 불러오기
import gradio as gr
from PIL import Image, ImageOps, ImageDraw, ImageFont
import os

# ---------------------------------------------------------
# 3) 사용 가능한 폰트를 찾는 함수
# ---------------------------------------------------------
# 코랩/리눅스 환경에서 자주 쓰이는 폰트 경로를 여러 개 시도합니다.
# 가장 먼저 찾은 폰트를 사용합니다.
def find_font(font_size):
    candidate_paths = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
    ]

    for path in candidate_paths:
        if os.path.exists(path):
            try:
                return ImageFont.truetype(path, font_size), path
            except:
                pass

    # 어떤 폰트도 못 찾으면 아주 작은 기본 폰트로 fallback
    return ImageFont.load_default(), "DEFAULT_FONT"

# ---------------------------------------------------------
# 4) 이미지 처리 함수 만들기
# ---------------------------------------------------------
def process_image(image, caption):
    """
    업로드한 이미지를 처리하는 함수

    입력(input):
        - image: 업로드한 이미지
        - caption: 이미지 위에 넣을 텍스트

    처리(process):
        - grayscale 이미지 만들기
        - 아래쪽 배너에 큰 자막 넣기

    출력(output):
        - 원본 이미지
        - 흑백 이미지
        - 자막이 들어간 이미지
        - 설명 문장
    """

    if image is None:
        return None, None, None, "Please upload an image."

    # 원본 이미지를 RGB로 통일
    original_img = image.convert("RGB")

    # 1) 흑백 이미지 만들기
    gray_img = ImageOps.grayscale(original_img)

    # 2) 캡션 이미지 만들기
    caption_img = original_img.copy()
    draw = ImageDraw.Draw(caption_img)

    img_width, img_height = caption_img.size

    # -----------------------------------------------------
    # 폰트 크기를 이미지 크기에 비례해서 설정
    # 최소 32 정도로 잡아 너무 작아지지 않게 합니다.
    # -----------------------------------------------------
    font_size = max(32, img_width // 15)

    # 실제 사용할 폰트 찾기
    font, font_path_used = find_font(font_size)

    # -----------------------------------------------------
    # 아래쪽 검정 배너 만들기
    # -----------------------------------------------------
    banner_height = max(100, img_height // 5)

    draw.rectangle(
        [(0, img_height - banner_height), (img_width, img_height)],
        fill="black"
    )

    # -----------------------------------------------------
    # 텍스트 크기 계산
    # -----------------------------------------------------
    bbox = draw.textbbox((0, 0), caption, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]

    # -----------------------------------------------------
    # 텍스트 중앙 정렬
    # -----------------------------------------------------
    x = (img_width - text_width) // 2
    y = img_height - banner_height + (banner_height - text_height) // 2

    # -----------------------------------------------------
    # 글자를 더 또렷하게 보이게 하기
    # 노란색 글자 + 검정 외곽선
    # -----------------------------------------------------
    draw.text(
        (x, y),
        caption,
        font=font,
        fill="yellow",
        stroke_width=3,
        stroke_fill="black"
    )

    # 설명 메시지
    if font_path_used == "DEFAULT_FONT":
        message = (
            "Warning ⚠️\n\n"
            "A proper large font was not found, so Pillow used the default tiny font.\n"
            "That is why the caption may still look very small."
        )
    else:
        message = (
            "Done! ✅\n\n"
            "Input → image + caption\n"
            "Function → grayscale + large caption banner\n"
            "Output → new images\n\n"
            f"Font used: {font_path_used}"
        )

    return original_img, gray_img, caption_img, message

# ---------------------------------------------------------
# 5) Gradio 앱 만들기
# ---------------------------------------------------------
demo = gr.Interface(
    fn=process_image,
    inputs=[
        gr.Image(type="pil", label="Upload an image"),
        gr.Textbox(label="Enter a caption", value="Hello!")
    ],
    outputs=[
        gr.Image(label="Original Image"),
        gr.Image(label="Grayscale Image"),
        gr.Image(label="Captioned Image"),
        gr.Textbox(label="Explanation")
    ],
    title="Image Processing App 🖼️",
    description="Upload an image and create grayscale and captioned versions."
)

# ---------------------------------------------------------
# 6) 앱 실행
# ---------------------------------------------------------
demo.launch()

### 결과가 의미하는 것 💡

이 앱은 이미지도 함수로 처리할 수 있다는 것을 보여 줍니다.

즉,
- 그림은 그냥 보는 자료가 아니라
- **변형 가능한 데이터**입니다.

영어수업에서는 flashcard, picture prompt, warm-up 자료로 연결할 수 있습니다.

# App 4. Picture Book App 🖼️🎧

이번에는 그림책 활동 예시입니다.

- 그림 1과 그림 2를 보여주고
- 그림 아래 버튼을 누르면
- 해당 문장을 TTS로 읽어 줍니다

즉,  
**이미지 + 텍스트 + 소리**  
를 결합한 멀티모달 앱입니다.

In [ ]:
# =========================================================
# Picture Book App with Buttons under Images
# =========================================================

# 1) 필요한 라이브러리 설치
!pip -q install gradio gTTS requests

# 2) 라이브러리 불러오기
import gradio as gr
from gtts import gTTS
import requests
import tempfile

# =========================================================
# 3) GitHub 이미지 raw 링크
# =========================================================
fig1_url = "https://raw.githubusercontent.com/junkyuhufs/Class2026spring/main/RedHat_Fig1.png"
fig2_url = "https://raw.githubusercontent.com/junkyuhufs/Class2026spring/main/RedHat_Fig2.png"

# =========================================================
# 4) 스토리 문장
# =========================================================
story1_text = "One windy day, Tom lost his red hat in the park. He ran after it, but the wind was too strong."
story2_text = "A small dog caught the hat in its mouth. The dog brought it back to Tom, and Tom smiled happily."

# =========================================================
# 5) 웹 이미지를 파일로 저장하는 함수
# =========================================================
def download_image(url, filename):
    response = requests.get(url)
    response.raise_for_status()
    with open(filename, "wb") as f:
        f.write(response.content)
    return filename

# 이미지 다운로드
fig1_path = download_image(fig1_url, "RedHat_Fig1.png")
fig2_path = download_image(fig2_url, "RedHat_Fig2.png")

# =========================================================
# 6) 텍스트를 mp3로 바꾸는 함수
# =========================================================
def make_tts(text):
    # 임시 mp3 파일 만들기
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    temp_file.close()

    # gTTS로 음성 생성
    tts = gTTS(text=text, lang="en")
    tts.save(temp_file.name)

    return temp_file.name

# =========================================================
# 7) 버튼을 누르면 실행될 함수
# =========================================================
def play_story1():
    audio_path = make_tts(story1_text)
    return story1_text, audio_path

def play_story2():
    audio_path = make_tts(story2_text)
    return story2_text, audio_path

# =========================================================
# 8) Gradio 앱 만들기
# =========================================================
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Picture Book App 🖼️🎧
        Click the button under each picture to hear the matching story sentence.

        This is a simple multimodal app:
        **image + text + sound**
        """
    )

    with gr.Row():
        with gr.Column():
            gr.Image(value=fig1_path, label="Figure 1", interactive=False)
            btn1 = gr.Button("▶ Play Story 1")

        with gr.Column():
            gr.Image(value=fig2_path, label="Figure 2", interactive=False)
            btn2 = gr.Button("▶ Play Story 2")

    # 출력 영역
    story_output = gr.Textbox(label="Story Sentence", lines=3)
    audio_output = gr.Audio(label="TTS Audio", type="filepath", autoplay=True)

    gr.Markdown(
        """
        ### Teaching idea 💡
        - picture-based storytelling
        - listening and matching
        - story retelling
        - young learner speaking activities
        """
    )

    # 버튼 클릭 시 함수 실행
    btn1.click(
        fn=play_story1,
        inputs=None,
        outputs=[story_output, audio_output]
    )

    btn2.click(
        fn=play_story2,
        inputs=None,
        outputs=[story_output, audio_output]
    )

# =========================================================
# 9) 앱 실행
# =========================================================
demo.launch(share=True)

### 결과가 의미하는 것 💡

이 앱은 그림과 문장을 연결하고,  
그 문장을 다시 소리로 들려줍니다.

즉,  
**image → text → sound**  
흐름을 보여 주는 간단한 멀티모달 앱입니다.

# App 5. Picture Book Order Quiz 🖼️🎧✅

이번에는 그림책을 퀴즈 앱으로 바꿉니다.

- 일부러 그림 2를 먼저 보여줍니다
- 각 그림의 문장을 들어봅니다
- 올바른 이야기 순서를 고릅니다
- 정답이면 “Well done!” 음성이 나옵니다
- 틀리면 “Try again.” 음성이 나옵니다
- 다시 하기 버튼도 있습니다

즉, 이 앱은  
**이미지 + 듣기 + 이해 + 퀴즈**  
를 결합한 수업용 앱입니다.

In [ ]:
# =========================================================
# Picture Book Order Quiz App
# with Feedback Audio + Reset Button
# =========================================================

# 1) 필요한 라이브러리 설치
!pip -q install gradio gTTS requests

# 2) 라이브러리 불러오기
import gradio as gr
from gtts import gTTS
import requests
import tempfile

# =========================================================
# 3) GitHub 이미지 raw 링크
# =========================================================
fig1_url = "https://raw.githubusercontent.com/junkyuhufs/Class2026spring/main/RedHat_Fig1.png"
fig2_url = "https://raw.githubusercontent.com/junkyuhufs/Class2026spring/main/RedHat_Fig2.png"

# =========================================================
# 4) 스토리 문장
# ---------------------------------------------------------
# 실제 이야기 순서는 story1 -> story2
# 하지만 퀴즈에서는 일부러 Figure 2를 먼저 보여줍니다.
# =========================================================
story1_text = "One windy day, Tom lost his red hat in the park. He ran after it, but the wind was too strong."
story2_text = "A small dog caught the hat in its mouth. The dog brought it back to Tom, and Tom smiled happily."

# =========================================================
# 5) 이미지 다운로드 함수
# =========================================================
def download_image(url, filename):
    response = requests.get(url)
    response.raise_for_status()
    with open(filename, "wb") as f:
        f.write(response.content)
    return filename

# 이미지 파일 저장
fig1_path = download_image(fig1_url, "RedHat_Fig1.png")
fig2_path = download_image(fig2_url, "RedHat_Fig2.png")

# =========================================================
# 6) 텍스트를 mp3로 바꾸는 함수
# =========================================================
def make_tts(text):
    """
    입력된 영어 문장을 TTS mp3 파일로 저장합니다.
    """
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    temp_file.close()

    tts = gTTS(text=text, lang="en")
    tts.save(temp_file.name)

    return temp_file.name

# =========================================================
# 7) 그림별 문장/음성 함수
# =========================================================
def play_story_for_fig2():
    """
    Figure 2 버튼을 누르면 실행됩니다.
    story2 문장과 오디오를 반환합니다.
    """
    audio_path = make_tts(story2_text)
    return story2_text, audio_path

def play_story_for_fig1():
    """
    Figure 1 버튼을 누르면 실행됩니다.
    story1 문장과 오디오를 반환합니다.
    """
    audio_path = make_tts(story1_text)
    return story1_text, audio_path

# =========================================================
# 8) 정답 확인 함수
# ---------------------------------------------------------
# 정답은 Figure 1 -> Figure 2
# 정답/오답에 따라 피드백 문장 + 피드백 오디오를 반환합니다.
# =========================================================
def check_order(answer):
    """
    학생이 고른 순서를 확인합니다.

    입력(input):
        - answer: 학생이 선택한 순서

    출력(output):
        - 결과 메시지
        - 피드백 오디오
    """
    correct_answer = "Figure 1 → Figure 2"

    if answer == correct_answer:
        feedback_text = "Well done! Figure 1 comes first, and Figure 2 comes next."
    else:
        feedback_text = "Try again. Think about what happened first in the story."

    feedback_audio = make_tts(feedback_text)
    return feedback_text, feedback_audio

# =========================================================
# 9) 다시 하기 함수
# ---------------------------------------------------------
# 선택값, 결과창, 오디오창 등을 초기화합니다.
# =========================================================
def reset_quiz():
    """
    퀴즈를 처음 상태로 되돌립니다.
    """
    return None, "", None, "", None

# =========================================================
# 10) Gradio 앱 만들기
# =========================================================
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Picture Book Order Quiz 🖼️🎧
        ## Quiz: Choose the correct order of the story

        Look at the two pictures.
        Listen to the sentences.
        Then choose the correct order.

        **Important:**
        The pictures are shown in a mixed order on purpose.
        """
    )

    gr.Markdown("### Step 1. Click the buttons to listen to each picture sentence.")

    # -----------------------------------------------------
    # 일부러 Figure 2를 먼저 보여줍니다.
    # -----------------------------------------------------
    with gr.Row():
        with gr.Column():
            gr.Image(value=fig2_path, label="Figure 2", interactive=False)
            btn2 = gr.Button("▶ Play Figure 2 Sentence")

        with gr.Column():
            gr.Image(value=fig1_path, label="Figure 1", interactive=False)
            btn1 = gr.Button("▶ Play Figure 1 Sentence")

    # 그림 문장/오디오 출력
    story_output = gr.Textbox(label="Story Sentence", lines=3)
    audio_output = gr.Audio(label="Story Audio", type="filepath", autoplay=True)

    # 버튼 연결
    btn2.click(
        fn=play_story_for_fig2,
        inputs=None,
        outputs=[story_output, audio_output]
    )

    btn1.click(
        fn=play_story_for_fig1,
        inputs=None,
        outputs=[story_output, audio_output]
    )

    # -----------------------------------------------------
    # 퀴즈 영역
    # -----------------------------------------------------
    gr.Markdown("### Step 2. Quiz: Choose the correct order of the story.")

    order_choice = gr.Radio(
        choices=[
            "Figure 1 → Figure 2",
            "Figure 2 → Figure 1"
        ],
        label="Choose the correct order"
    )

    with gr.Row():
        check_button = gr.Button("Check Answer")
        reset_button = gr.Button("Reset / Try Again")

    result_output = gr.Textbox(label="Quiz Result")
    feedback_audio_output = gr.Audio(label="Feedback Audio", type="filepath", autoplay=True)

    # 정답 확인 연결
    check_button.click(
        fn=check_order,
        inputs=order_choice,
        outputs=[result_output, feedback_audio_output]
    )

    # 다시 하기 연결
    reset_button.click(
        fn=reset_quiz,
        inputs=None,
        outputs=[order_choice, result_output, feedback_audio_output, story_output, audio_output]
    )

    gr.Markdown(
        """
        ### Teaching ideas 💡
        - listening and ordering
        - story sequence activities
        - speaking: retell the story in order
        - young learner picture-based comprehension
        """
    )

# =========================================================
# 11) 앱 실행
# =========================================================
demo.launch(share=True)

### 결과가 의미하는 것 💡

이 앱은 단순한 그림 보기 앱이 아니라  
**퀴즈 + 피드백 + 다시 하기** 기능이 있는 학습 앱입니다.

즉, 그림책 활동도  
디지털 퀴즈 앱으로 바꿀 수 있다는 것을 보여줍니다.

# 오늘 수업 정리 ✅

오늘 우리는 Gradio를 이용해  
파이썬 함수가 앱으로 바뀌는 예시를 여러 가지 보았습니다.

### 핵심 구조
- **입력(input)**
- **함수(function)**
- **출력(output)**

### 오늘 본 앱들
1. 텍스트 분석 앱  
2. 사운드(TTS) 앱  
3. 이미지 처리 앱  
4. 그림책 앱  
5. 그림책 퀴즈 앱  

즉, 텍스트, 소리, 이미지, 퀴즈까지  
모두 앱으로 구현할 수 있다는 것을 체험한 것입니다.

# Data Types and Multimodal Teaching for English Teachers 🌍📚🎧🖼️

## Why does this matter? 🎯

In the AI era, English teachers do more than just use textbooks and ready-made materials.  
They can also **analyze**, **transform**, and **create** teaching materials using different types of data.

A very important idea is this:

**Different data types make different kinds of teaching possible.**

For example:
- text data helps with reading and writing
- sound data helps with listening and pronunciation
- image data helps with speaking and vocabulary
- table/numeric data helps with comparison, feedback, and visualization

When we combine two or more data types, we call it **multimodal**.

---

# 1. Text Data ✍️

## What is text data?
Text data includes:
- words
- sentences
- reading passages
- student writing
- YouTube transcripts
- song lyrics

## What can English teachers do with text data?
- count words
- find frequent words
- make a word cloud
- check lexical diversity
- check readability
- make cloze quizzes
- see words in context
- analyze student writing

## Example classroom uses
- reading passage analysis
- vocabulary teaching
- writing feedback
- key expression extraction
- text simplification/adaptation

## Example apps
- Reading Passage Analyzer
- Vocabulary Profile App
- Cloze Quiz Generator
- Student Writing Analyzer

## Why is this important?
Text is the most basic data type in English teaching.  
Teachers should not only **read texts**, but also **analyze, transform, and reuse them**.

---

# 2. Sound Data 🎧

## What is sound data?
Sound data includes:
- TTS audio
- student voice recordings
- pronunciation files
- YouTube audio
- podcast audio
- songs

## What can English teachers do with sound data?
- make listening materials with TTS
- provide pronunciation models
- create shadowing materials
- make dictation activities
- compare different English accents
- let students listen and repeat

## Example classroom uses
- pronunciation practice
- listening comprehension
- dictation
- shadowing
- world Englishes awareness

## Example apps
- Text-to-Speech App
- World Englishes Pronunciation App
- Dictation Helper App
- Listening Practice App

## Why is this important?
English teaching is not only about written language.  
It is also about **sound**, **pronunciation**, and **listening**.  
So sound data is essential.

---

# 3. Image Data 🖼️

## What is image data?
Image data includes:
- pictures
- photos
- flashcards
- textbook illustrations
- student drawings
- web images

## What can English teachers do with image data?
- show picture prompts
- make flashcards
- add words or questions to images
- create picture description tasks
- make story sequence activities
- prepare warm-up visuals

## Example classroom uses
- vocabulary teaching
- speaking prompts
- writing prompts
- story retelling
- young learner activities

## Example apps
- Image Caption App
- Flashcard Maker
- Picture Prompt App
- Story Image Viewer

## Why is this important?
Images are especially useful for:
- young learners
- speaking warm-ups
- vocabulary presentation
- storytelling activities

---

# 4. Table / Numeric Data 📊

## What is table/numeric data?
This type of data includes:
- student scores
- survey responses
- word counts
- text difficulty indicators
- learning logs

## What can English teachers do with table/numeric data?
- compare student scores
- compare writing length
- summarize survey results
- visualize class performance
- organize text features in tables
- track progress over time

## Example classroom uses
- comparing groups
- giving feedback
- checking participation
- analyzing writing length
- visualizing survey results

## Example apps
- Score Dashboard
- Writing Comparison App
- Survey Viewer
- Text Feature Dashboard

## Why is this important?
Teachers increasingly need to make decisions based on data.  
This is part of **data-informed teaching**.

---

# 5. What is Multimodal? 🔗

## Definition
**Multimodal** means using **two or more data types together**.

Examples:
- text + sound
- image + text
- video + text
- image + text + sound
- video + transcript + sound

In other words, multimodal teaching means that learning is supported through more than one mode.

---

# 6. Multimodal Possibilities in English Teaching 🚀

## A. Text + Sound
### Example
- type a sentence and hear it with TTS
- turn a reading passage into listening material
- convert transcript text into speech

### Classroom use
- listening + reading integration
- pronunciation practice
- shadowing
- dictation

---

## B. Image + Text
### Example
- add labels to an image
- show a picture and ask students to describe it
- create vocabulary cards with images

### Classroom use
- vocabulary teaching
- speaking prompts
- writing prompts
- picture-based activities

---

## C. Video + Text
### Example
- load a YouTube transcript
- save subtitles as text
- extract key expressions from video subtitles

### Classroom use
- authentic input
- listening script creation
- lyric-based lessons
- discourse analysis

---

## D. Text + Sound + Image
### Example
- upload 2 pictures
- display a short story
- read the story with TTS

### Classroom use
- story retelling
- picture-based listening
- integrated speaking + listening
- young learner storytelling

---

## E. Video + Text + Sound
### Example
- load a YouTube transcript
- save it as a text file
- extract top words
- turn part of it into TTS audio

### Classroom use
- turning videos into reading materials
- vocabulary extraction
- listening + reading integration
- key expression practice

---

# 7. Why should English teachers care? 💡

## 1. Different data types support different language skills
- **Text** → reading, writing, vocabulary
- **Sound** → listening, pronunciation
- **Image** → speaking, vocabulary, warm-up
- **Table/Numeric** → comparison, feedback, evaluation

## 2. Multimodal teaching makes lessons richer
The same content can be shown:
- as text
- as sound
- as image
- or as a combination of all three

## 3. AI makes it easier to transform one type of data into another
For example:
- text → sound
- video subtitles → text
- image + text → teaching prompt
- transcript → vocabulary list + TTS

## 4. Teachers can become designers, not just users
In the AI era, teachers do not have to remain only material users.  
They can also become **material designers** and **tool creators**.

---

# 8. A Simple Summary Table ✅

| Data Type | Examples | What Teachers Can Do | Example Apps |
|---|---|---|---|
| Text | passages, writing, transcripts | reading analysis, vocabulary, writing feedback | Reading Analyzer, Cloze App |
| Sound | TTS, recordings, audio | listening, pronunciation, dictation | TTS App, Pronunciation App |
| Image | pictures, photos, flashcards | speaking prompts, vocabulary, warm-up | Flashcard App, Picture Prompt App |
| Table/Numeric | scores, survey, word counts | comparison, feedback, visualization | Score Dashboard, Writing Comparison App |
| Multimodal | text+sound, image+text, video+text | integrated skills teaching | Storybook App, YouTube Transcript App |

---

# 9. Final Message for Students 🌟

The goal is not just to learn Python code.

The bigger goal is this:

**English teachers in the AI era should understand different data types and know how to turn them into teaching materials, activities, and apps.**

In other words:
- text can become analysis
- text can become sound
- images can become prompts
- videos can become transcripts
- multiple data types can become multimodal lessons

That is one important meaning of **computational thinking for English teachers**.